In [3]:
!pip install pandas sqlalchemy psycopg2-binary

  Using cached pandas-2.3.3-cp39-cp39-win_amd64.whl (11.4 MB)
  Using cached sqlalchemy-2.0.52-cp39-cp39-win_amd64.whl (2.2 MB)
  Using cached psycopg2_binary-2.9.12-cp39-cp39-win_amd64.whl (2.8 MB)
  Using cached tzdata-2026.3-py2.py3-none-any.whl (348 kB)
  Using cached pytz-2026.3.post1-py2.py3-none-any.whl (508 kB)
  Using cached greenlet-3.2.5.tar.gz (191 kB)
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
    Preparing wheel metadata: started
    Preparing wheel metadata: finished with status 'done'
Failed to build greenlet


  ERROR: Command errored out with exit status 1:
   command: 'c:\Users\ARIEL\OneDrive\Documents\PSD\.venv\Scripts\python.exe' 'c:\Users\ARIEL\OneDrive\Documents\PSD\.venv\lib\site-packages\pip\_vendor\pep517\in_process\_in_process.py' build_wheel 'C:\Users\ARIEL\AppData\Local\Temp\tmpalgcpd7h'
       cwd: C:\Users\ARIEL\AppData\Local\Temp\pip-install-za9vpugi\greenlet_01fca0c4fea44349b54b4583014dfdc8
  Complete output (112 lines):
  running bdist_wheel
  running build
  running build_py
  creating build\lib.win-amd64-cpython-39\greenlet
  copying src\greenlet\__init__.py -> build\lib.win-amd64-cpython-39\greenlet
  creating build\lib.win-amd64-cpython-39\greenlet\platform
  copying src\greenlet\platform\__init__.py -> build\lib.win-amd64-cpython-39\greenlet\platform
  creating build\lib.win-amd64-cpython-39\greenlet\tests
  copying src\greenlet\tests\fail_clearing_run_switches.py -> build\lib.win-amd64-cpython-39\greenlet\tests
  copying src\greenlet\tests\fail_cpp_exception.py -> buil

In [ ]:
import pandas as pd
from sqlalchemy import create_engine, text
from sqlalchemy.types import Float

# 1. Baca ketiga file CSV
df_no2 = pd.read_csv('NO2_KAMAL-UTM.csv')
df_so2 = pd.read_csv('SO2_KAMAL-UTM.csv')
df_co = pd.read_csv('CO_KAMAL-UTM.csv')

# 2. Gabungkan
df_gabung = pd.merge(df_no2, df_so2, on=['date', 'feature_index'], how='outer')
df_gabung = pd.merge(df_gabung, df_co, on=['date', 'feature_index'], how='outer')

print("Data berhasil digabung! Berikut 5 baris pertamanya:")
print(df_gabung.head())

# 3. Koneksi Database
DATABASE_URI = "masukan url aiven anda"
engine = create_engine(DATABASE_URI)
table_name = 'kualitas_udara_kamal'

# 4. Upload dengan tipe double precision
try:
    with engine.connect() as conn:
        conn.execute(text(f"DROP TABLE IF EXISTS {table_name};"))
        conn.commit()
        print(f"Tabel lama '{table_name}' berhasil dihapus.")
    
    df_gabung.to_sql(
        name=table_name,
        con=engine,
        if_exists='replace',
        index=False,
        dtype={
            'NO2': Float(precision=53),
            'SO2': Float(precision=53),
            'CO':  Float(precision=53)
        }
    )
    print(f"Tabel '{table_name}' berhasil diupload dengan tipe double precision!")

except Exception as e:
    print(f"Terjadi kesalahan: {e}")

Data berhasil digabung! Berikut 5 baris pertamanya:
                       date  feature_index       NO2       SO2        CO
0  2025-08-24T00:00:00.000Z              0       NaN       NaN       NaN
1  2025-08-25T00:00:00.000Z              0  0.000020  0.000272  0.028240
2  2025-08-26T00:00:00.000Z              0  0.000035 -0.002125       NaN
3  2025-08-27T00:00:00.000Z              0  0.000086  0.000169  0.030417
4  2025-08-28T00:00:00.000Z              0  0.000010  0.000109  0.025137
Tabel lama 'kualitas_udara_kamal' berhasil dihapus.
Tabel 'kualitas_udara_kamal' berhasil diupload dengan tipe double precision!


In [8]:
df_gabung.to_csv('data_gabungan_lengkap.csv', index=False)

print("File CSV berhasil diekspor!")

File CSV berhasil diekspor!


In [1]:
!pip install scipy

  Using cached numpy-2.0.2-cp39-cp39-win_amd64.whl (15.9 MB)


You should consider upgrading via the 'c:\Users\ARIEL\OneDrive\Documents\PSD\.venv\Scripts\python.exe -m pip install --upgrade pip' command.


In [6]:
import pandas as pd
import numpy as np
from scipy.stats import skew, kurtosis

# Membaca data time series
df = pd.read_csv("data_time_series_20.csv")
ts = df["Nilai"].values

# Perubahan antar waktu
diff = np.diff(ts)

# Trend
x = np.arange(len(ts))
trend = np.polyfit(x, ts, 1)[0]

# Autocorrelation lag 1
autocorr_lag1 = np.corrcoef(ts[:-1], ts[1:])[0, 1]

# RMS
rms = np.sqrt(np.mean(ts ** 2))

# Energy
energy = np.sum(ts ** 2)

# 14 fitur
fitur = {
    "Mean": np.mean(ts),
    "Median": np.median(ts),
    "Std Dev": np.std(ts, ddof=1),
    "Variance": np.var(ts, ddof=1),
    "Minimum": np.min(ts),
    "Maximum": np.max(ts),
    "Range": np.max(ts) - np.min(ts),
    "Skewness": skew(ts),
    "Kurtosis": kurtosis(ts),
    "Mean Absolute Change": np.mean(np.abs(diff)),
    "Trend": trend,
    "Autocorrelation Lag 1": autocorr_lag1,
    "RMS": rms,
    "Energy": energy
}

# Menampilkan hasil
for i, (nama, nilai) in enumerate(fitur.items(), 1):
    print(f"{i}. {nama}: {nilai:.4f}")

ModuleNotFoundError: No module named 'pandas'